# D11 KRX 유가증권시장 일별매매정보 API 검증

## 목적
한국거래소(KRX) Open API를 이용하여
유가증권시장 상장주식의 일별 매매정보를 조회할 수 있는지 검증한다.

### 테스트 대상
- 기업: 삼성전자
- 종목코드: 005930
- 시장: KOSPI

### 검증 항목
1. KRX Open API 연결 가능 여부
2. 특정 거래일의 유가증권시장 전체 데이터 조회 가능 여부
3. 삼성전자 종목 데이터 추출 가능 여부
4. 시가·고가·저가·종가·거래량 확인
5. D06 금융위원회 주식시세정보와 동일 날짜 값 대조 가능 여부

### 특징
- 2010-01-04 이후 데이터 제공
- 인증키는 HTTP Header의 AUTH_KEY로 전달

In [1]:
import requests
import pandas as pd
from getpass import getpass

pd.set_option("display.max_columns", None)

In [5]:
AUTH_KEY = getpass("KRX AUTH_KEY 입력: ")

BASE_URL = (
    "https://data-dbg.krx.co.kr/"
    "svc/apis/sto/stk_bydd_trd"
)

TEST_DATE = "20261012"

KRX AUTH_KEY 입력:  ········


In [6]:
def get_kospi_daily_trade(auth_key, base_date):

    headers = {
        "AUTH_KEY": auth_key
    }

    params = {
        "basDd": base_date
    }

    try:
        response = requests.get(
            BASE_URL,
            headers=headers,
            params=params,
            timeout=60
        )

        print("HTTP 상태코드:", response.status_code)

        if response.status_code != 200:
            print("API 호출 실패")
            print(response.text[:2000])
            return pd.DataFrame()

        data = response.json()

        print("응답 key:", data.keys())

        rows = data.get("OutBlock_1", [])

        if not rows:
            print("조회 결과가 없습니다.")
            return pd.DataFrame()

        return pd.DataFrame(rows)

    except requests.exceptions.ReadTimeout:
        print("서버 응답시간 초과")
        return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print("API 요청 오류:", e)
        return pd.DataFrame()

In [7]:
df = get_kospi_daily_trade(
    auth_key=AUTH_KEY,
    base_date=TEST_DATE
)

print("조회 건수:", len(df))

if not df.empty:
    print("\n전체 컬럼:")
    print(df.columns.tolist())

    display(df.head())

HTTP 상태코드: 200
응답 key: dict_keys(['OutBlock_1'])
조회 건수: 941

전체 컬럼:
['BAS_DD', 'ISU_CD', 'ISU_NM', 'MKT_NM', 'SECT_TP_NM', 'TDD_CLSPRC', 'CMPPREVDD_PRC', 'FLUC_RT', 'TDD_OPNPRC', 'TDD_HGPRC', 'TDD_LWPRC', 'ACC_TRDVOL', 'ACC_TRDVAL', 'MKTCAP', 'LIST_SHRS']


,BAS_DD,ISU_CD,ISU_NM,MKT_NM,SECT_TP_NM,TDD_CLSPRC,CMPPREVDD_PRC,FLUC_RT,TDD_OPNPRC,TDD_HGPRC,TDD_LWPRC,ACC_TRDVOL,ACC_TRDVAL,MKTCAP,LIST_SHRS
0,20261012,095570,AJ네트웍스,KOSPI,,4240,-15,-0.35,4260,4260,4190,79951,337844496,191871698160,45252759
1,20261012,006840,AK홀딩스,KOSPI,,6600,-70,-1.05,6670,6860,6510,3661,24058250,87433902600,13247561
2,20261012,027410,BGF,KOSPI,,3740,-60,-1.58,3800,3820,3695,76711,288154970,357980798340,95716791
3,20261012,282330,BGF리테일,KOSPI,,128600,2200,1.74,125100,132600,124800,47139,6109532950,2222710311600,17283906
4,20261012,138930,BNK금융지주,KOSPI,,15130,-180,-1.18,15170,15500,14950,1380400,20897599490,4642485432630,306839751


In [9]:
samsung = df[
    df["ISU_NM"].astype(str).str.strip() == "삼성전자"
].copy()

display(samsung)

,BAS_DD,ISU_CD,ISU_NM,MKT_NM,SECT_TP_NM,TDD_CLSPRC,CMPPREVDD_PRC,FLUC_RT,TDD_OPNPRC,TDD_HGPRC,TDD_LWPRC,ACC_TRDVOL,ACC_TRDVAL,MKTCAP,LIST_SHRS
456,20261012,005930,삼성전자,KOSPI,,250500,-11500,-4.39,260000,262500,249500,19771955,5018226038736,1464492791304000,5846278608


In [10]:
display(
    samsung[
        [
            "BAS_DD",
            "ISU_CD",
            "ISU_NM",
            "MKT_NM",
            "TDD_OPNPRC",
            "TDD_HGPRC",
            "TDD_LWPRC",
            "TDD_CLSPRC",
            "ACC_TRDVOL",
            "ACC_TRDVAL",
            "MKTCAP"
        ]
    ]
)

,BAS_DD,ISU_CD,ISU_NM,MKT_NM,TDD_OPNPRC,TDD_HGPRC,TDD_LWPRC,TDD_CLSPRC,ACC_TRDVOL,ACC_TRDVAL,MKTCAP
456,20261012,005930,삼성전자,KOSPI,260000,262500,249500,250500,19771955,5018226038736,1464492791304000


In [11]:
numeric_cols = [
    "TDD_OPNPRC",
    "TDD_HGPRC",
    "TDD_LWPRC",
    "TDD_CLSPRC",
    "ACC_TRDVOL",
    "ACC_TRDVAL",
    "MKTCAP",
    "LIST_SHRS"
]

for col in numeric_cols:
    if col in samsung.columns:
        samsung[col] = pd.to_numeric(
            samsung[col].astype(str).str.replace(",", ""),
            errors="coerce"
        )

In [12]:
print("===== D11 검증 결과 =====")

print("API 데이터 존재:", not df.empty)
print("유가증권시장 조회 건수:", len(df))
print("삼성전자 검색 결과:", len(samsung))

if not samsung.empty:
    print(
        "조회 기준일:",
        samsung["BAS_DD"].iloc[0]
    )

    print(
        "삼성전자 종가:",
        samsung["TDD_CLSPRC"].iloc[0]
    )

    print(
        "삼성전자 거래량:",
        samsung["ACC_TRDVOL"].iloc[0]
    )

===== D11 검증 결과 =====
API 데이터 존재: True
유가증권시장 조회 건수: 941
삼성전자 검색 결과: 1
조회 기준일: 20261012
삼성전자 종가: 250500
삼성전자 거래량: 19771955
